# Tren topik / sentimen dan laporan

Gunakan waktu lama untuk fit, waktu baru untuk audit. Model topik tetap; bandingkan proporsi dan jumlah, jangan menyimpulkan sebab-akibat dari timeline.

Kode lengkap dari pipeline dan snippet asli. Ikuti urutan tahap dan ubah konfigurasi awal sesuai data. Alternatif tersedia pada cheatsheet utama.

## Setup & konfigurasi

Satu-satunya sel yang wajib diubah. Jalankan paling awal.

In [ ]:
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import importlib.metadata as md

def env_report(models_dir=None):
    import torch
    for p in ("torch", "transformers", "scikit-learn", "pandas", "numpy", "sentence-transformers", "lightgbm"):
        try:
            print(p, md.version(p))
        except md.PackageNotFoundError:
            print(p, "TIDAK ADA")
    gpu = torch.cuda.is_available()
    print("cuda", gpu, torch.cuda.get_device_name(0) if gpu else "")
    if gpu:
        print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "| bf16", torch.cuda.is_bf16_supported())
    if models_dir and Path(models_dir).exists():
        print("model lokal:", sorted(p.name for p in Path(models_dir).iterdir() if p.is_dir()))
    return gpu

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# ===== jalankan =====
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
MODELS = Path(os.environ.get("COMP_ROOT", r"C:\gemastik")) / "models"
SEED = 42
seed_everything(SEED)
env_report(MODELS)

DATA, TEXT, TIME = "train.csv", "text", "time"
K_TOPIC = 8
ISSUE = "keamanan_pangan"                        # isu yang dipantau di garis waktu


## Muat data

Baca file, otomatis mencoba encoding, simpan parquet.

In [ ]:
from pathlib import Path
import pandas as pd

def read_any(path, cache=True, **kw):
    """df = read_any('train.csv'). Lain kali otomatis membaca train.parquet."""
    p = Path(path)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kw)
    if cache and p.with_suffix(".parquet").exists():
        return pd.read_parquet(p.with_suffix(".parquet"))
    if p.suffix in (".jsonl", ".json"):
        return pd.read_json(p, lines=p.suffix == ".jsonl", **kw)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p, **kw)
    sep = kw.pop("sep", "\t" if p.suffix == ".tsv" else ",")
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, on_bad_lines="warn", **kw)
            break
        except UnicodeDecodeError:
            continue
    if cache:
        df.to_parquet(p.with_suffix(".parquet"), index=False)
    return df

# ===== jalankan =====
df = read_any(DATA)
print(df.shape)


## EDA

Profil, duplikat, dan baris tak bermakna.

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict

def _nk(s):
    return s.fillna("").astype(str).str.lower().str.replace(r"\W+", " ", regex=True).str.strip()

def eda_report(train, test=None, text="text", target=None, time=None, n_show=8, sample=20000, seed=42):
    """Kembalikan dict ringkasan berisi saran dan rekomendasi max_words. target: nama kolom atau daftar kolom."""
    tips, out = [], {}
    print("=== PROFIL ===\ntrain", train.shape, "| test", None if test is None else test.shape)
    print(pd.DataFrame({"dtype": train.dtypes.astype(str), "null": train.isna().mean().round(4), "nunique": train.nunique()}).to_string())
    if test is not None:
        print("kolom hanya di train:", sorted(set(train.columns) - set(test.columns)))
    tx = train[text].fillna("").astype(str)
    print("\n=== CONTOH ===")
    for t in train[text].sample(min(n_show, len(train)), random_state=seed):
        print(" -", str(t)[:180])
    key = _nk(train[text])
    out["kosong"], out["duplikat"] = round(float(key.eq("").mean()), 4), round(float(key.duplicated().mean()), 4)
    print("\n=== KUALITAS ===\nkosong", out["kosong"], "| duplikat ternormalisasi", out["duplikat"], "| punya URL", round(tx.str.contains(r"https?://").mean(), 3),
          "| mention", round(tx.str.contains(r"@\w").mean(), 3), "| tanda baca akhir kalimat", round(tx.str.contains(r"[.!?]").mean(), 3))
    print("teks paling sering:", key.value_counts().head(3).to_dict())
    if out["duplikat"] > .01:
        tips.append("Duplikat > 1%: pakai group fold dan laporkan skor random serta grup.")
    w = tx.str.split().str.len()
    q = w.quantile([.5, .9, .95, .99]).round(0)
    out["max_words"] = int(q[.95])
    print("\n=== PANJANG (kata) ===\n", q.to_dict(), "| char p50", int(tx.str.len().median()))
    if q[.5] < 5:
        tips.append("Teks sangat pendek: utamakan TF-IDF karakter.")
    if q[.95] > 256:
        tips.append("Teks panjang: pertimbangkan head+tail dan chunk.")
    if target is not None:
        print("\n=== TARGET ===")
        cols = target if isinstance(target, (list, tuple)) else [target]
        if len(cols) > 1:
            Y = train[cols].to_numpy()
            print(pd.DataFrame({"pct": train[cols].mean().round(4)}).T.to_string(), "\nlabel per baris", round(float(Y.sum(1).mean()), 3))
        else:
            s = train[cols[0]]
            if pd.api.types.is_numeric_dtype(s) and s.nunique() > 20:
                print(s.describe().round(3).to_dict(), "skew", round(s.skew(), 3))
                if s.skew() > 1 and s.min() >= 0:
                    tips.append("Target berekor panjang: latih pada log1p.")
            else:
                vc = s.value_counts(dropna=False)
                print((vc / len(s)).round(4).to_dict())
                out["rasio_imbalance"] = round(float(vc.max() / max(vc.min(), 1)), 1)
                if out["rasio_imbalance"] > 20:
                    tips.append("Imbalance parah: metrik macro-F1/PR-AUC, decision layer dari OOF.")
                if s.nunique() <= 50 and s.notna().all():
                    v = TfidfVectorizer(max_features=30000, min_df=3, sublinear_tf=True)
                    X = v.fit_transform(tx.iloc[:sample])
                    m = LogisticRegression(C=4, max_iter=1000).fit(X, s.iloc[:sample].astype(str))
                    names, coef = v.get_feature_names_out(), m.coef_ if m.coef_.shape[0] > 1 else np.vstack([-m.coef_[0], m.coef_[0]])
                    print("kata khas (cek artefak):")
                    for j, c in enumerate(m.classes_):
                        print(f"  {c}:", [names[i] for i in coef[j].argsort()[::-1][:10]])
    cv = CountVectorizer(token_pattern=r"(?u)\b\w+\b", binary=True)
    X = cv.fit_transform(tx.sample(min(sample, len(tx)), random_state=seed))
    df_ = np.asarray(X.sum(0)).ravel()
    print("\n=== KOSAKATA ===\nvocab", X.shape[1], "| kata muncul sekali", round(float((df_ == 1).mean()), 3))
    if test is not None and text in test:
        te = test[text].fillna("").astype(str)
        vt = CountVectorizer(token_pattern=r"(?u)\b\w+\b").fit(te.sample(min(sample, len(te)), random_state=seed))
        cnt = np.asarray(vt.transform(te.sample(min(sample, len(te)), random_state=seed)).sum(0)).ravel()
        oov = np.array([x not in cv.vocabulary_ for x in vt.get_feature_names_out()])
        out["oov"] = round(float(cnt[oov].sum() / max(cnt.sum(), 1)), 3)
        tk = set(key)
        out["overlap"] = round(float(_nk(te).isin(tk).mean()), 4)
        print("OOV token test", out["oov"], "| teks test ada di train", out["overlap"])
        if out["oov"] > .2:
            tips.append("OOV tinggi: TF-IDF karakter atau FastText.")
        n = min(5000, len(tx), len(te))
        t2 = list(tx.sample(n, random_state=seed)) + list(te.sample(n, random_state=seed))
        lab = np.r_[np.zeros(n), np.ones(n)]
        Xa = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000).fit_transform(t2)
        out["adversarial_auc"] = round(float(roc_auc_score(lab, cross_val_predict(LogisticRegression(max_iter=500), Xa, lab, cv=3, method="predict_proba")[:, 1])), 4)
        print("AUC adversarial", out["adversarial_auc"], "| panjang p50 test", int(te.str.split().str.len().median()))
        if out["adversarial_auc"] > .7:
            tips.append("Train dan test berbeda: cek kata pembeda, pertimbangkan split waktu/sumber.")
    if time and time in train:
        t = pd.to_datetime(train[time], errors="coerce")
        print("\n=== WAKTU ===\nrentang", t.min(), "s.d.", t.max(), "| gagal parse", round(float(t.isna().mean()), 3))
        if test is not None and time in test:
            tt = pd.to_datetime(test[time], errors="coerce")
            print("test", tt.min(), "s.d.", tt.max())
            if tt.min() > t.median():
                tips.append("Test lebih baru dari sebagian besar train: pakai time split.")
    out["saran"] = tips
    print("\n=== SARAN ===", *(tips or ["Tidak ada peringatan khusus."]), sep="\n - ")
    return out

import re
import numpy as np
import pandas as pd

def text_flags(texts, min_chars=3, max_chars=None):
    s = pd.Series(list(texts)).fillna("").astype(str)
    n = s.str.len().clip(lower=1)
    alpha = s.str.count(r"[^\W\d_]") / n
    d = pd.DataFrame({"kosong": s.str.strip().eq(""), "pendek": n < min_chars, "non_huruf": alpha < .5, "sticker": s.str.contains(r"\[sticker\]", case=False, regex=True),
                      "panjang": n > (max_chars or n.quantile(.999) * 1.0 + 1), "url_saja": s.str.fullmatch(r"\s*https?://\S+\s*")})
    print(d.mean().round(4).to_string())
    return d

def domain_noise(texts, terms):
    """Saring kata seperti judi atau slot dengan Aho-Corasick berbatas kata. Audit hasilnya."""
    import ahocorasick
    A = ahocorasick.Automaton()
    for t in terms:
        A.add_word(f" {t.lower()} ", t)
    A.make_automaton()
    return np.array([any(True for _ in A.iter(" " + " ".join(re.findall(r"\w+", str(x).lower())) + " ")) for x in texts])

# ===== jalankan =====
report = eda_report(df, None, TEXT, time=TIME if TIME in df else None)
flags = text_flags(df[TEXT])


## Cleaning: volume asli vs copypasta

Pembeda laporan: pisahkan copypasta.

In [ ]:
import re
import numpy as np
import pandas as pd

_PSEUDO = {"", "nan", "none", "null", "-", "--", "n/a", "na", "[sticker]"}

def _groups_minhash(keys, threshold=.8, num_perm=64, shingle=5):
    from datasketch import MinHash, MinHashLSH
    lsh, par = MinHashLSH(threshold=threshold, num_perm=num_perm), list(range(len(keys)))
    def find(x):
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    for i, t in enumerate(keys):
        m = MinHash(num_perm=num_perm)
        for j in range(max(len(t) - shingle + 1, 1)):
            m.update(t[j:j + shingle].encode())
        for k in lsh.query(m):
            par[find(i)] = find(int(k))
        lsh.insert(str(i), m)
    return pd.factorize(np.array([find(i) for i in range(len(keys))]))[0]

def clean_dataset(df, text="text", target=None, dedup=True, near_dup=False, conflict=None, drop_empty=True, near_threshold=.8):
    """Kembalikan df bersih dengan kolom _group untuk fold. conflict: 'majority', 'drop', atau None."""
    n0 = len(df)
    df = df.copy()
    df[text] = df[text].fillna("").astype(str).str.strip()
    if drop_empty:
        df = df[~df[text].str.lower().isin(_PSEUDO)]
    key = df[text].str.lower().str.replace(r"\W+", " ", regex=True).str.strip()
    df["_group"] = pd.factorize(key)[0]
    if near_dup:
        df["_group"] = _groups_minhash(key.tolist(), near_threshold)
    n_conf = 0
    if target is not None and conflict:
        g = df.groupby(key)[target]
        bad = g.transform("nunique") > 1
        n_conf = int(bad.sum())
        if conflict == "majority":
            df[target] = g.transform(lambda s: s.mode().iloc[0])
        elif conflict == "drop":
            df, key = df[~bad], key[~bad]
    if dedup:
        # Konflik label tetap utuh dan satu grup; tidak memilih label pertama diam-diam.
        keep = ~pd.DataFrame({'key':key,'target':df[target]}).duplicated() if target is not None and conflict is None else ~key.duplicated()
        df = df[keep]
    print(f"baris {n0} -> {len(df)} | konflik label {n_conf} | grup {df._group.nunique()}")
    return df.reset_index(drop=True)

def clean_test(df, text="text"):
    """Test tidak dibuang: hanya isi kosong."""
    df = df.copy()
    df[text] = df[text].fillna("").astype(str)
    return df

# ===== jalankan =====
d = clean_dataset(df, TEXT)
print("copypasta:", round(1 - len(d) / len(df), 4))


## Preprocessing

Preset topic.

In [ ]:
import html
import re
import unicodedata

SLANG = {"yg": "yang", "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "tdk": "tidak", "bgt": "banget", "tp": "tapi", "klo": "kalau", "kalo": "kalau",
         "udh": "sudah", "dah": "sudah", "dpt": "dapat", "jg": "juga", "jd": "jadi", "bnyk": "banyak", "karna": "karena", "krn": "karena", "utk": "untuk", "jgn": "jangan",
         "org": "orang", "dr": "dari", "lgi": "lagi", "lg": "lagi", "bnr": "benar", "sdh": "sudah", "dgn": "dengan", "aja": "saja"}
STOPWORDS = set("""ada adalah agar akan aku anda antara apa apakah atau bagi bahwa banyak beberapa begitu bila bisa dalam dan dari dengan di dia dua hal hanya harus ia ini itu
jadi jika juga kami kamu karena ke kepada ketika kita lagi lain lebih maka masih mereka mungkin nya oleh pada para saat saja sama sampai sangat satu saya sebagai sebuah sedang
sehingga sekali selalu semua seperti setelah sudah supaya tapi telah tentang tersebut tetapi untuk yaitu yang""".split())   # sengaja tanpa tidak, bukan, jangan, belum, kurang
NEGATION = "tidak|tak|bukan|jangan|belum|kurang|tanpa"
_MOJIBAKE = re.compile("[\u00c3\u00c2\u00e2\u00f0]")

PRESETS = {
    # keep = biarkan, token = ganti penanda, remove = hapus
    "none":        dict(unicode=False, html=False, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "tfidf":       dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="keep", emoji="keep", number="keep", lower=True, repeat=True, punct="keep", slang=False, negation=False, stopword=False),
    "social":      dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="word", emoji="text", number="keep", lower=True, repeat=True, punct="keep", slang=True, negation=False, stopword=False),
    "topic":       dict(unicode=True, html=True, url="remove", email="remove", phone="remove", mention="remove", hashtag="word", emoji="remove", number="remove", lower=True, repeat=True, punct="remove", slang=True, negation=False, stopword=True),
    "transformer": dict(unicode=True, html=True, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "indobertweet": dict(unicode=True, html=True, url="HTTPURL", email="keep", phone="keep", mention="@USER", hashtag="keep", emoji="text", number="keep", lower=True, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
}

def _sub(text, pattern, how, token):
    return text if how == "keep" else re.sub(pattern, " " if how == "remove" else f" {token if how == 'token' else how} ", text)

def preprocess(text, preset="tfidf", slang_dict=SLANG, stopwords=STOPWORDS, **override):
    """preprocess('Halo http://a.id', 'social'). Ubah satu langkah: preprocess(t, 'tfidf', emoji='remove')."""
    c = {**PRESETS[preset], **override}
    t = "" if text is None or text != text else str(text)
    if c["unicode"]:
        if _MOJIBAKE.search(t):                         # teks rusak encoding
            import ftfy
            t = ftfy.fix_text(t)
        t = unicodedata.normalize("NFKC", t).replace("​", "")
    if c["html"]:
        t = re.sub(r"<[^>]+>", " ", html.unescape(t))
    t = _sub(t, r"[\w.+-]+@[\w-]+\.[\w.-]+", c["email"], "EMAIL")          # email sebelum mention
    t = _sub(t, r"https?://\S+|www\.\S+", c["url"], "URL")
    t = _sub(t, r"(?<!\d)(?:\+?62|0)8\d{7,11}(?!\d)", c["phone"], "TELEPON")
    t = _sub(t, r"@\w+", c["mention"], "USER")
    if c["hashtag"] == "word":
        t = re.sub(r"#(\w+)", r" \1 ", t)
    elif c["hashtag"] != "keep":
        t = _sub(t, r"#\w+", c["hashtag"], "HASHTAG")
    if c["emoji"] != "keep":
        import emoji
        if c["emoji"] == "id":                          # emoji jadi kata Indonesia
            from indoNLP.preprocessing import emoji_to_words
            t = emoji_to_words(t, delimiter=(" ", " "))
        else:
            t = emoji.demojize(t, delimiters=(" ", " ")) if c["emoji"] == "text" else emoji.replace_emoji(t, " " if c["emoji"] == "remove" else " EMOJI ")
    t = _sub(t, r"\d+(?:[.,]\d+)*", c["number"], "ANGKA")
    if c["lower"]:
        keep = {m: m for m in ("HTTPURL", "@USER", "URL", "EMAIL", "TELEPON", "USER", "ANGKA", "EMOJI", "HASHTAG") if m in t}
        t = t.lower()
        for m in keep:
            t = re.sub(r"(?<!\w)" + re.escape(m.lower()) + r"(?!\w)", m, t)
    if c["repeat"]:
        t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    if c["punct"] == "remove":
        t = re.sub(r"[^\w\s]", " ", t)
    if c["slang"]:
        t = re.sub(r"\b\w+\b", lambda m: slang_dict.get(m.group(0).lower(), m.group(0)), t)
    if c["negation"]:
        t = re.sub(rf"\b({NEGATION})\s+(\w+)", r"\1_\2", t, flags=re.I)
    if c["stopword"]:
        t = " ".join(w for w in t.split() if w.lower() not in stopwords)
    return re.sub(r"\s+", " ", t).strip()

def preprocess_corpus(texts, preset="tfidf", stem=False, slang_csv=None, **override):
    """Versi cepat: teks identik hanya diproses sekali, stem per kata unik. Kembalikan list."""
    slang = dict(SLANG)
    if slang_csv:
        import pandas as pd
        lex = pd.read_csv(slang_csv)
        slang.update(dict(zip(lex["slang"].astype(str), lex["formal"].astype(str))))
    texts = list(texts)
    cache = {t: preprocess(t, preset, slang_dict=slang, **override) for t in {x if x == x else None for x in texts}}
    out = [cache[t if t == t else None] for t in texts]
    if stem:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        st = StemmerFactory().create_stemmer()
        m = {w: st.stem(w) for w in {w for d in out for w in d.split()}}
        out = [" ".join(m[w] for w in d.split()) for d in out]
    return out

def text_versions(df, text="text", slang_csv=None):
    """Tiga versi agar efek normalisasi bisa diuji: text_raw, text_clean (makna utuh), text_norm (slang dinormalkan)."""
    d = df.copy()
    d["text_raw"] = d[text]
    d["text_clean"] = preprocess_corpus(d[text], "tfidf")
    d["text_norm"] = preprocess_corpus(d[text], "social", slang_csv=slang_csv)
    return d

# ===== jalankan =====
docs = preprocess_corpus(d[TEXT], "topic")


## Topik & aspek

NMF untuk tema, leksikon untuk aspek. Audit presisi aturan pada 30 sampel.

In [ ]:
import numpy as np
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def _top(comp, names, k):
    return [[names[i] for i in c.argsort()[::-1][:k]] for c in comp]

def fit_nmf(docs, k=20, top=10, min_df=5, max_df=.5, stop=None, seed=42):
    v = TfidfVectorizer(min_df=min_df, max_df=max_df, stop_words=stop, sublinear_tf=True, token_pattern=r"(?u)\b[^\d\W]{2,}\b")
    X = v.fit_transform(docs)
    m = NMF(k, init="nndsvd", max_iter=400, random_state=seed)
    W = m.fit_transform(X)
    return W.argmax(1), _top(m.components_, v.get_feature_names_out(), top), W

def fit_lda(docs, k=20, top=10, min_df=5, max_df=.5, stop=None, seed=42):
    v = CountVectorizer(min_df=min_df, max_df=max_df, stop_words=stop, token_pattern=r"(?u)\b[^\d\W]{2,}\b")
    X = v.fit_transform(docs)
    m = LatentDirichletAllocation(k, learning_method="online", batch_size=4096, random_state=seed)
    W = m.fit_transform(X)
    return W.argmax(1), _top(m.components_, v.get_feature_names_out(), top), W

import re
import numpy as np

def topic_diversity(topics, top=10):
    w = [x for t in topics for x in t[:top]]
    return round(len(set(w)) / max(len(w), 1), 4)

def npmi_coherence(topics, docs, top=10, eps=1e-12):
    """Kemunculan bersama pada tingkat dokumen. Tanpa paket tambahan."""
    sets = [set(re.findall(r"\w+", str(d).lower())) for d in docs]
    vocab = {w for t in topics for w in t[:top] for w in w.split()}
    inv = {w: {i for i, s in enumerate(sets) if w in s} for w in vocab}
    n = len(sets)
    out = []
    for t in topics:
        ws = [w for w in t[:top] if inv.get(w)]
        vals = []
        for i in range(len(ws)):
            for j in range(i + 1, len(ws)):
                a, b = inv[ws[i]], inv[ws[j]]
                p = len(a & b) / n
                vals.append(-1.0 if p == 0 else np.log((p + eps) / (len(a) / n * len(b) / n)) / -np.log(p + eps))
        out.append(float(np.mean(vals)) if vals else 0.0)
    return round(float(np.mean(out)), 4), [round(x, 4) for x in out]

def cv_coherence(topics, docs, top=10):
    """c_v gensim. Lebih lambat."""
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel
    toks = [re.findall(r"\w+", str(d).lower()) for d in docs]
    return CoherenceModel(topics=[t[:top] for t in topics], texts=toks, dictionary=Dictionary(toks), coherence="c_v", processes=1).get_coherence()

import re
import ahocorasick
import numpy as np
import pandas as pd

MBG_ENTITIES = {
    "MBG": ["mbg", "makan bergizi gratis", "makan bergizi", "makan gratis", "makan siang gratis", "program makan gratis", "embege"],
    "BGN": ["bgn", "badan gizi nasional", "badan gizi"],
    "SPPG": ["sppg", "satuan pelayanan pemenuhan gizi", "dapur mbg", "dapur sppg"],
    "Kemenkes": ["kemenkes", "kementerian kesehatan", "menkes"], "Kejagung": ["kejagung", "kejaksaan agung"], "KPK": ["kpk"],
    "DPR": ["dpr", "banggar"], "JPPI": ["jppi"], "KPAI": ["kpai"], "Ombudsman": ["ombudsman"], "ICW": ["icw"], "CISDI": ["cisdi"],
    "Prabowo": ["prabowo", "wowo"], "Gibran": ["gibran"], "Sudaryono": ["sudaryono"], "Dadan Hindayana": ["dadan", "dadan hindayana"], "Nanik Deyang": ["nanik"], "Purbaya": ["purbaya"],
}
MBG_ASPECTS = {
    "keamanan_pangan": ["keracunan", "keracunannya", "racun", "beracun", "basi", "busuk", "berjamur", "jamuran", "belatung", "ulat", "ulet", "klb", "e coli", "salmonella", "kadaluarsa", "kedaluwarsa", "tidak layak", "gak layak", "ga layak", "mual", "muntah", "diare", "pusing", "sakit perut", "dilarikan", "puskesmas", "rumah sakit", "korban", "bau", "mentah"],
    "kualitas_menu": ["menu", "ompreng", "ayam", "nasi", "sayur", "susu", "burger", "spaghetti", "spageti", "roti", "telur", "telor", "tempe", "ikan", "kacang", "buah", "pisang", "jeruk", "lauk", "enak", "hambar", "porsi", "ultra processed", "upf", "sosis", "nugget"],
    "gizi_kesehatan": ["gizi", "bergizi", "ahli gizi", "nutrisi", "stunting", "protein", "kalori", "akg", "obesitas", "sehat", "kesehatan", "dokter"],
    "anggaran": ["anggaran", "apbn", "rapbn", "pajak", "triliun", "pagu", "efisiensi", "pemborosan", "refocusing", "dana", "buang uang", "uang rakyat", "biaya", "per porsi", "15 ribu", "10 ribu", "10rb", "15rb"],
    "korupsi_tata_kelola": ["korupsi", "koruptor", "korup", "nepotisme", "kolusi", "markup", "mark up", "tersangka", "kejagung", "jual beli titik", "yayasan", "calo", "transparan", "transparansi", "proyek", "bancakan", "berjamaah", "kpk", "tata kelola", "pungli"],
    "distribusi_dapur": ["sppg", "dapur", "distribusi", "pengiriman", "terlambat", "telat", "kapasitas", "suspend", "sanitasi", "slhs", "sop", "relawan", "mitra", "juknis", "ditutup", "penerima", "belum dapat", "tidak merata", "gak merata", "merata"],
    "pendidikan_guru": ["guru", "honorer", "murid", "siswa", "sekolah", "kantin", "santri", "pendidikan", "gaji guru", "biaya sekolah", "sekolah gratis"],
    "ekonomi_umkm": ["umkm", "lapangan kerja", "pekerja", "pedagang", "jualan", "catering", "katering", "petani", "peternak", "nelayan", "susu lokal"],
    "politik": ["prabowo", "wowo", "gibran", "dpr", "tni", "polri", "janji kampanye", "militer", "purbaya", "zulhas", "sudaryono", "nanik", "dadan", "presiden", "pemerintah", "menteri"],
    "media_hoaks": ["hoax", "hoaks", "buzzer", "influencer", "narasi", "digoreng", "komdigi", "klarifikasi", "fitnah"],
    "penolakan": ["stop", "setop", "hentikan", "bubar", "bubarkan", "batalkan", "hapus", "ganti uang", "tolak", "menolak"],
    "minta_perbaikan": ["evaluasi", "perbaiki", "perbaikan", "benahi", "awasi", "diawasi", "pengawasan", "tolong", "seharusnya", "harusnya"],
    "dukungan": ["bagus", "setuju", "mendukung", "dukung", "program bagus", "bermanfaat", "terima kasih", "makasih", "alhamdulillah", "lanjutkan", "semoga"],
}
MBG_EVENTS = {"2025-01-06": "Peluncuran MBG", "2025-09-24": "KLB Bandung Barat", "2026-06-02": "Pimpinan BGN dicopot", "2026-07-22": "Kepala BGN baru", "2026-10-29": "Laporan 100 hari"}
MBG_FOOD = sorted({w for w in MBG_ASPECTS["kualitas_menu"] if w not in ("menu", "ompreng", "enak", "hambar", "porsi", "ultra processed", "upf")})
MBG_SYMPTOM = ["mual", "muntah", "diare", "pusing", "sakit perut", "demam", "lemas"]
MBG_INCIDENT = ["keracunan", "basi", "busuk", "berjamur", "belatung", "ulat", "klb", "kadaluarsa", "kedaluwarsa"]

_SUFFIX = re.compile(r"(?:nya|ny|ku|mu|lah|kah)?(?![0-9a-z])")

def build_matcher(lex):
    """lex: {kelas: [istilah]}. Kembalikan automaton, dipakai find_terms."""
    a = ahocorasick.Automaton()
    for key, terms in lex.items():
        for t in terms:
            t = t.lower()
            old = a.get(t) if t in a else (t, [])
            a.add_word(t, (t, old[1] + [key]))
    a.make_automaton()
    return a

def find_terms(text, matcher):
    """Istilah utuh dalam teks. Akhiran -nya, -ku, -mu ikut cocok: 'menunya' cocok 'menu'."""
    t = str(text).lower()
    out = []
    for end, (term, keys) in matcher.iter(t):
        start = end - len(term) + 1
        if (start == 0 or not t[start - 1].isalnum()) and _SUFFIX.match(t, end + 1):
            out.append((term, keys))
    return out

def tag_terms(texts, lex=None):
    """DataFrame bool: baris x kelas. Teks identik hanya diproses sekali."""
    lex = lex or MBG_ASPECTS
    keys, idx = list(lex), {k: i for i, k in enumerate(lex)}
    m = build_matcher(lex)
    texts = ["" if t is None or t != t else str(t) for t in texts]
    cache = {}
    for t in set(texts):
        row = np.zeros(len(keys), dtype=bool)
        for _, ks in find_terms(t, m):
            row[[idx[k] for k in ks]] = True
        cache[t] = row
    return pd.DataFrame(np.vstack([cache[t] for t in texts]) if texts else np.zeros((0, len(keys)), bool), columns=keys)

def term_hits(texts, lex=None):
    """Jumlah baris per istilah. Istilah nol hit dibuang, istilah terlalu sering diaudit."""
    lex = lex or MBG_ASPECTS
    m = build_matcher(lex)
    c = {}
    for t in texts:
        for term in {x[0] for x in find_terms(t, m)}:
            c[term] = c.get(term, 0) + 1
    rows = [(k, t, c.get(t.lower(), 0)) for k, v in lex.items() for t in v]
    return pd.DataFrame(rows, columns=["kelas", "istilah", "hit"]).sort_values(["kelas", "hit"], ascending=[True, False]).reset_index(drop=True)

def audit_aspects(texts, H, n=30, seed=42):
    """{kelas: contoh acak}. Baca dan hitung presisinya sebelum dipakai di slide."""
    texts = pd.Series(list(texts))
    return {k: texts[H[k].to_numpy()].drop_duplicates().sample(min(n, int(H[k].sum())), random_state=seed).tolist() for k in H if H[k].any()}

def aspect_sentiment(H, sentiment):
    """Tabel aspek x sentimen. Satu baris dihitung di semua aspeknya."""
    s = np.asarray(sentiment)
    return pd.DataFrame({k: pd.Series(s[H[k].to_numpy()]).value_counts() for k in H}).T.fillna(0).astype(int)

# ===== jalankan =====
lab, words, _ = fit_nmf(docs, k=K_TOPIC, min_df=2)
print("npmi", npmi_coherence(words, docs)[0], "diversity", topic_diversity(words))
for i, w in enumerate(words):
    print(i, w[:8])
H = tag_terms(d[TEXT])
samples = audit_aspects(d[TEXT], H, 30)
print("istilah nol hit:", term_hits(d[TEXT]).query("hit == 0").istilah.tolist()[:30])
d["aspek"] = np.where(H.any(axis=1), H.columns[H.to_numpy().argmax(1)], "lainnya")
d["topik"] = lab


## Garis waktu

Lonjakan dan proporsi topik per minggu.

In [ ]:
import numpy as np
import pandas as pd

def psi(a, b, bins=10, eps=1e-4):
    a, b = np.asarray(a, float), np.asarray(b, float)
    edges = np.unique(np.quantile(a, np.linspace(0, 1, bins + 1)))
    if len(edges) < 2:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(a, edges)[0] / len(a) + eps
    q = np.histogram(b, edges)[0] / len(b) + eps
    return round(float(((q - p) * np.log(q / p)).sum()), 4)

def psi_table(train_df, test_df):
    c = [x for x in train_df.columns if x in test_df.columns and pd.api.types.is_numeric_dtype(train_df[x])]
    return pd.Series({x: psi(train_df[x].dropna(), test_df[x].dropna()) for x in c}).sort_values(ascending=False)

def volume_timeline(ts, freq="D", z=3.0):
    """Volume per periode dan tanda lonjakan. Pakai untuk garis waktu slide."""
    t = pd.to_datetime(pd.Series(list(ts)), errors="coerce").dropna()
    v = t.dt.to_period(freq).value_counts().sort_index()
    med = v.rolling(14, min_periods=3, center=True).median().fillna(v.median())
    mad = (v - med).abs().median() or 1
    return pd.DataFrame({"n": v, "lonjakan": (v - med) / (1.4826 * mad) > z})

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ruptures as rpt

def _period(times, freq):
    t = pd.to_datetime(pd.Series(list(times)), errors="coerce", utc=True, format="mixed").dt.tz_localize(None)
    return t, t.dt.to_period(freq)

def issue_timeline(times, mask=None, freq="W", min_total=30):
    """Tabel per periode: total, isu, proporsi. Periode di bawah min_total ditandai agar tidak dibaca."""
    t, p = _period(times, freq)
    m = np.ones(len(t), bool) if mask is None else np.asarray(mask, bool)
    tl = pd.DataFrame({"total": p[t.notna()].value_counts(), "isu": p[t.notna() & m].value_counts()}).fillna(0).astype(int).sort_index()
    tl["proporsi"] = (tl.isu / tl.total).round(4)
    tl["cukup"] = tl.total >= min_total
    return tl

def change_points(tl, col="proporsi", penalty=None, min_size=2):
    """Titik perubahan pada periode yang cukup datanya. Kembalikan daftar periode awal segmen baru."""
    s = tl[tl.cukup][col] if "cukup" in tl else tl[col]
    if len(s) < 2 * min_size + 1:
        return []
    x = s.to_numpy(float).reshape(-1, 1)
    pen = penalty if penalty is not None else 3 * np.log(len(x)) * max(float(np.var(x)), 1e-9)
    bk = rpt.Pelt(model="l2", min_size=min_size, jump=1).fit(x).predict(pen=pen)
    return [s.index[i] for i in bk[:-1]]

def share_over_time(times, labels, freq="M", min_total=30):
    """Proporsi tiap topik per periode. Periode kecil dibuang."""
    t, p = _period(times, freq)
    ok = t.notna().to_numpy()
    c = pd.crosstab(p[ok].to_numpy(), np.asarray(labels)[ok])
    c = c[c.sum(1) >= min_total]
    return c.div(c.sum(1), axis=0).round(3)

def plot_issue_timeline(tl, path="tren_isu.png", title="Porsi isu per periode", events=None, cps=None):
    d = tl[tl.cukup] if "cukup" in tl else tl
    x = d.index.to_timestamp()
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.bar(x, d.total, width=(x[1] - x[0]) * .8 if len(x) > 1 else 1, color="#d0d5dd")
    ax.set_ylabel("volume")
    a2 = ax.twinx()
    a2.plot(x, d.proporsi, color="#b42318", lw=2)
    a2.set_ylabel("porsi isu")
    for c in cps or []:
        ax.axvline(c.to_timestamp(), color="#0f766e", lw=1.5)
    for when, name in (events or {}).items():
        w = pd.Timestamp(when)
        if len(x) and x[0] <= w <= x[-1]:
            ax.axvline(w, color="#667085", ls="--", lw=1)
            ax.text(w, d.total.max(), name, fontsize=8, rotation=90, va="top")
    ax.set_title(title)
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()
    return path

def bertopic_over_time(docs, times, embeddings, nr_bins=12, min_topic_size=30):
    """BERTopic pada embedding yang sudah ada. Pakai sampel 20-50 ribu baris; RAM 500 ribu belum teruji."""
    from bertopic import BERTopic
    tm = BERTopic(min_topic_size=min_topic_size, calculate_probabilities=False, low_memory=True, verbose=False)
    tm.fit(docs, embeddings)
    return tm, tm.topics_over_time(docs, list(pd.to_datetime(pd.Series(list(times)), errors="coerce", utc=True, format="mixed").dt.tz_localize(None)), nr_bins=nr_bins)

# ===== jalankan =====
if TIME in d:
    v = volume_timeline(d[TIME], "W")
    print(v.query("lonjakan").head().to_string())
    tl = issue_timeline(d[TIME], H[ISSUE], "W")
    cps = change_points(tl)
    print("titik perubahan:", cps)
    t = pd.to_datetime(d[TIME], errors="coerce").dt.to_period("M")
    print(pd.crosstab(t, d.topik, normalize="index").round(2).tail(6).to_string())


## Grafik & slide

Data tidak boleh keluar: simpan gambar dan masukkan selagi lomba berjalan.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

def _save(path):
    plt.tight_layout(); plt.savefig(path, dpi=160); plt.close()
    return path

def plot_heatmap(table, path="heatmap.png", title="Aspek x sentimen", fmt=".0f", normalize=False):
    t = table.div(table.sum(1), axis=0) if normalize else table
    plt.figure(figsize=(8, max(3, .4 * len(t))))
    sns.heatmap(t, annot=True, fmt=".2f" if normalize else fmt, cmap="YlGnBu", cbar=False)
    plt.title(title)
    return _save(path)

def plot_confusion(y, pred, names=None, path="confusion.png"):
    c = confusion_matrix(y, pred, normalize="true")
    return plot_heatmap(pd.DataFrame(c, index=names, columns=names), path, "Confusion matrix (per baris)", fmt=".2f")

def plot_timeline(ts, path="timeline.png", events=None, freq="W", title="Volume per minggu"):
    """events: {'YYYY-MM-DD': 'nama kejadian'}"""
    t = pd.to_datetime(pd.Series(list(ts)), errors="coerce").dropna()
    v = t.dt.to_period(freq).value_counts().sort_index()
    x = v.index.to_timestamp()
    plt.figure(figsize=(10, 3.6)); plt.plot(x, v.values, color="#0f766e")
    for d, name in (events or {}).items():
        plt.axvline(pd.Timestamp(d), color="#b42318", ls="--", lw=1); plt.text(pd.Timestamp(d), v.max(), name, fontsize=8, rotation=90, va="top")
    plt.title(title)
    return _save(path)

def plot_bars(series, path="bars.png", title="Distribusi", top=20):
    s = pd.Series(series).sort_values().tail(top)
    plt.figure(figsize=(8, max(3, .35 * len(s)))); plt.barh(s.index.astype(str), s.values, color="#0f766e"); plt.title(title)
    return _save(path)

from pptx import Presentation
from pptx.util import Inches, Pt

def build_slides(slides, out="hasil.pptx", template=None):
    """slides: list dict {title, bullets, image}. Satu slide per dict."""
    prs = Presentation(template) if template else Presentation()
    for s in slides:
        sl = prs.slides.add_slide(prs.slide_layouts[1])
        sl.shapes.title.text = s.get("title", "")
        body = sl.placeholders[1].text_frame
        body.clear()
        for i, b in enumerate(s.get("bullets", [])):
            p = body.paragraphs[0] if i == 0 else body.add_paragraph()
            p.text = str(b); p.font.size = Pt(20)
        if s.get("image"):
            sl.shapes.add_picture(s["image"], Inches(.6), Inches(3.2), height=Inches(3.8))
    prs.save(out)
    return out

# ===== jalankan =====
figs = [plot_bars(d.aspek.value_counts(), "aspek.png", "Aspek setelah dedup")]
if TIME in d:
    figs.append(plot_issue_timeline(tl, "timeline.png", f"Porsi {ISSUE} per minggu", MBG_EVENTS, cps))
build_slides([{"title": "Peta aspek", "bullets": [f"{len(d)} baris unik dari {len(df)}"], "image": figs[0]}], "hasil.pptx")
d.to_csv("submission.csv", index=False)
